# AutoScientist Tool-Caller — QLoRA fine-tune (Kaggle, one-click)

Produces a **real** LoRA adapter from the published dataset, computes the **real held-out number** with the repo's eval, and publishes weights to Hugging Face + Kaggle. Nothing fabricated.

**Before running:** in the right sidebar set **Accelerator = GPU T4 x2** (or P100) and **Internet = On**. Then Run All.

In [ ]:
# 1. GPU check + dependencies
!nvidia-smi -L
!pip -q install "transformers>=4.46" "peft>=0.13" "trl>=0.11" bitsandbytes accelerate datasets huggingface_hub "kagglehub>=1.0"

In [ ]:
# 2. Get the code
!git clone https://github.com/ankit25bcs10610/adaption-ai-lab.git
%cd adaption-ai-lab

In [ ]:
# 3. Auth — EITHER paste tokens here, OR use Add-ons → Secrets (HF_TOKEN, KAGGLE_API_TOKEN)
import os
HF_TOKEN = "hf_..."            # <-- your HF write token
KAGGLE_API_TOKEN = "KGAT_..."  # <-- your Kaggle access token

# Optional: pull from Kaggle Secrets instead of pasting
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    HF_TOKEN = s.get_secret("HF_TOKEN") or HF_TOKEN
    KAGGLE_API_TOKEN = s.get_secret("KAGGLE_API_TOKEN") or KAGGLE_API_TOKEN
except Exception:
    pass
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN
print("tokens set:", HF_TOKEN[:6] + "...", KAGGLE_API_TOKEN[:5] + "...")

### Base model
Free Kaggle T4/P100 → keep the default `Qwen/Qwen2.5-Coder-3B-Instruct`. The AutoScientist spec's `mistralai/Mixtral-8x7B-Instruct-v0.1` (46.7B) needs an A100-80GB — change `BASE` only if you have one.

In [ ]:
# 4. Train — produces the adapter and pushes it to HF
BASE = "Qwen/Qwen2.5-Coder-3B-Instruct"
!python scripts/train_lora.py \
  --base $BASE \
  --dataset pandeyankit84/autoscientist-toolcaller-dataset \
  --out out/lora_adapter \
  --push-to pandeyankit84/autoscientist-toolcaller

In [ ]:
# 5. Real held-out number (eligibility gate): base vs fine-tuned on the repo's test set
!bash scripts/finalize.sh MODEL=$BASE ADAPTER=out/lora_adapter
print("\n===== HEADLINE =====")
!cat HEADLINE.txt

In [ ]:
# 6. Publish weights to Kaggle (HF was pushed in cell 4)
!python -m autoscientist_toolcaller.release preflight    --dir out/lora_adapter
!python -m autoscientist_toolcaller.release kaggle-model --slug pandeyankit99/autoscientist-toolcaller --dir out/lora_adapter

### 7. Send back to finish
- the `HEADLINE.txt` line (base % → fine-tuned %)
- confirm `https://huggingface.co/pandeyankit84/autoscientist-toolcaller` and the Kaggle model now show weight files

Then the cards/site are finalized and only the two human steps remain: **post** (tag @adaption_ai) + **submit the Part 2 form**.